# PEECO Survivorship Presentation

This is the Jupyter Notebook used to produce the charts that are in Gabriel Virrey's part of the
presentation.




## Setting Up

In [1]:
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine, text

db_user = 'deserving0397'  # replace with the username from your docker-compose.yml configuration
db_url = 'localhost:5432'
db_name = 'psada_postgres'

with open('../db/password', 'r') as pwd_file:
    db_password = pwd_file.read().strip()

db_string = f'postgresql://{db_user}:{db_password}@{db_url}/{db_name}'

engine = create_engine(db_string);

## City of Manila Cohort Survivorship

In [2]:
query_male = """
             with bucketed as ( select width_bucket(age_years::int, 1, 121, 24) as sort_index
                                from vw_vsr_deaths_joined
                                where sex = 'Male'
                                  and death_year = 2024
                                  and age_years is not null
                                  and pod_level = 'City'
                                  and pod_name = 'City of Manila' )
             select sort_index
                  , case when sort_index = 0 then '0'
                         else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text end as x
                  , count(*)                                                                                  as d_x
             from bucketed
             group by 1, 2
             order by 1 asc; \
             """

with engine.connect() as conn:
    df_male = pd.read_sql_query(text(query_male), conn)

n_0 = df_male['d_x'].sum()

# l_x (Surviving)
df_male['l_x_raw'] = n_0 - df_male['d_x'].cumsum().shift(1, fill_value=0)

# Survivorship (l_x / n0) - formatting as a percentage string to match the worksheet
df_male['l_x_percent'] = ((df_male['l_x_raw'] / n_0) * 100).round(2).astype(str) + '%'

# q_x (Age-specific mortality) - formatting as a percentage string
df_male['q_x'] = ((df_male['d_x'] / df_male['l_x_raw']) * 100).round(2).astype(str) + '%'

# L_x = (l_x + l_x+1) / 2
df_male['l_x_next'] = df_male['l_x_raw'].shift(-1, fill_value=0)
df_male['L_x'] = (df_male['l_x_raw'] + df_male['l_x_next']) / 2

# T_x = Sum of L_x from age x to last age
df_male['T_x'] = df_male['L_x'].iloc[::-1].cumsum().iloc[::-1]

# e_x = T_x / L_x (Life Expectancy)
df_male['e_x'] = (df_male['T_x'] / df_male['L_x']).round(5)

# Format to match the worksheet columns perfectly and drop the sorting index
final_table_male = df_male[['x', 'd_x', 'l_x_percent', 'l_x_raw', 'q_x', 'L_x', 'T_x', 'e_x']]
print(final_table_male.to_string(index=False))

     x  d_x l_x_percent  l_x_raw    q_x     L_x      T_x      e_x
     0  563      100.0%    11691  4.82% 11409.5 136182.5 11.93589
   1-5  113      95.18%    11128  1.02% 11071.5 124773.0 11.26975
  6-10   72      94.22%    11015  0.65% 10979.0 113701.5 10.35627
 11-15   89       93.6%    10943  0.81% 10898.5 102722.5  9.42538
 16-20  186      92.84%    10854  1.71% 10761.0  91824.0  8.53304
 21-25  260      91.25%    10668  2.44% 10538.0  81063.0  7.69245
 26-30  359      89.03%    10408  3.45% 10228.5  70525.0  6.89495
 31-35  446      85.96%    10049  4.44%  9826.0  60296.5  6.13642
 36-40  484      82.14%     9603  5.04%  9361.0  50470.5  5.39157
 41-45  703       78.0%     9119  7.71%  8767.5  41109.5  4.68885
 46-50  857      71.99%     8416 10.18%  7987.5  32342.0  4.04908
 51-55 1080      64.66%     7559 14.29%  7019.0  24354.5  3.46980
 56-60 1260      55.42%     6479 19.45%  5849.0  17335.5  2.96384
 61-65 1381      44.64%     5219 26.46%  4528.5  11486.5  2.53649
 66-70 139

In [3]:
query_female = """
               with bucketed as ( select width_bucket(age_years::int, 1, 121, 24) as sort_index
                                  from vw_vsr_deaths_joined
                                  where sex = 'Female'
                                    and death_year = 2024
                                    and age_years is not null
                                    and pod_level = 'City'
                                    and pod_name = 'City of Manila' )
               select sort_index
                    , case when sort_index = 0 then '0'
                           else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text end as x
                    , count(*)                                                                                  as d_x
               from bucketed
               group by 1, 2
               order by 1 asc; \
               """

with engine.connect() as conn:
    df_female = pd.read_sql_query(text(query_female), conn)

n_0 = df_female['d_x'].sum()

# l_x (Surviving)
df_female['l_x_raw'] = n_0 - df_female['d_x'].cumsum().shift(1, fill_value=0)

# Survivorship (l_x / n0) - formatting as a percentage string to match the worksheet
df_female['l_x_percent'] = ((df_female['l_x_raw'] / n_0) * 100).round(2).astype(str) + '%'

# q_x (Age-specific mortality) - formatting as a percentage string
df_female['q_x'] = ((df_female['d_x'] / df_female['l_x_raw']) * 100).round(2).astype(str) + '%'

# L_x = (l_x + l_x+1) / 2
df_female['l_x_next'] = df_female['l_x_raw'].shift(-1, fill_value=0)
df_female['L_x'] = (df_female['l_x_raw'] + df_female['l_x_next']) / 2

# T_x = Sum of L_x from age x to last age
df_female['T_x'] = df_female['L_x'].iloc[::-1].cumsum().iloc[::-1]

# e_x = T_x / L_x (Life Expectancy)
df_female['e_x'] = (df_female['T_x'] / df_female['L_x']).round(5)

# Format to match the worksheet columns perfectly and drop the sorting index
final_table_female = df_female[['x', 'd_x', 'l_x_percent', 'l_x_raw', 'q_x', 'L_x', 'T_x', 'e_x']]
print(final_table_female.to_string(index=False))

     x  d_x l_x_percent  l_x_raw    q_x    L_x      T_x      e_x
     0  470      100.0%     9505  4.94% 9270.0 120653.5 13.01548
   1-5  107      95.06%     9035  1.18% 8981.5 111383.5 12.40144
  6-10   57      93.93%     8928  0.64% 8899.5 102402.0 11.50649
 11-15   78      93.33%     8871  0.88% 8832.0  93502.5 10.58679
 16-20  121      92.51%     8793  1.38% 8732.5  84670.5  9.69602
 21-25  151      91.24%     8672  1.74% 8596.5  75938.0  8.83360
 26-30  196      89.65%     8521   2.3% 8423.0  67341.5  7.99495
 31-35  243      87.59%     8325  2.92% 8203.5  58918.5  7.18212
 36-40  301      85.03%     8082  3.72% 7931.5  50715.0  6.39412
 41-45  439      81.86%     7781  5.64% 7561.5  42783.5  5.65807
 46-50  537      77.24%     7342  7.31% 7073.5  35222.0  4.97943
 51-55  727      71.59%     6805 10.68% 6441.5  28148.5  4.36987
 56-60  756      63.95%     6078 12.44% 5700.0  21707.0  3.80825
 61-65  922      55.99%     5322 17.32% 4861.0  16007.0  3.29294
 66-70  997      46.29%  

In [16]:
# We use the raw survivor count ('l_x_raw') for the Y-axis
fig = px.line(
    final_table_male,
    x='x',  # The age brackets (0, 1-5, 6-10...)
    y='l_x_raw',  # The remaining survivors out of your starting population
    title='2024 Male Cohort Survivorship Curve (City of Manila)',
    labels={
        'x': 'Age Interval',
        'l_x_raw': 'Number of Survivors (lx)'
    },
    log_y=True,
    markers=True  # Adds distinct dots to each data point
)

# Standard formatting to make it look professional
fig.update_traces(fill='tozeroy', line_color='blue')
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45  # Tilts the age labels so they don't overlap
)

fig.show()

In [17]:
# We use the raw survivor count ('l_x_raw') for the Y-axis
fig = px.line(
    final_table_female,
    x='x',  # The age brackets (0, 1-5, 6-10...)
    y='l_x_raw',  # The remaining survivors out of your starting population
    title='2024 Female Cohort Survivorship Curve (City of Manila)',
    labels={
        'x': 'Age Interval',
        'l_x_raw': 'Number of Survivors (lx)'
    },
    log_y=True,
    markers=True  # Adds distinct dots to each data point
)

# Standard formatting to make it look professional
fig.update_traces(fill='tozeroy', line_color='red')
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45  # Tilts the age labels so they don't overlap
)

fig.show()

In [6]:
import pandas as pd
import plotly.express as px
from sqlalchemy import text

# 1. SQL Query: Fetch death brackets for BOTH sexes
query_both = """
             with bucketed as ( select sex, width_bucket(age_years::int, 1, 101, 20) as sort_index
                                from vw_vsr_deaths_joined
                                where death_year = 2024
                                  and age_years is not null
                                  and sex in ('Male', 'Female')
                                  and pod_name = 'City of Manila' )
             select sex
                  , sort_index
                  , case when sort_index = 0 then '0'
                         else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text end as x
                  , count(*)                                                                                  as d_x
             from bucketed
             group by 1, 2, 3
             order by 1, 2 asc; \
             """

with engine.connect() as conn:
    df_both = pd.read_sql_query(text(query_both), conn)

# 1. Sort the dataframe to ensure the math flows perfectly from Age 0 upwards
df_both = df_both.sort_values(by=['sex', 'sort_index'])

# 2. Total starting population per sex (n_0)
# .transform('sum') broadcasts the total deaths for 'Male' and 'Female' to every row
df_both['n_0'] = df_both.groupby('sex')['d_x'].transform('sum')

# 3. Calculate cumulative deaths UP TO the current age bracket
df_both['cum_deaths'] = df_both.groupby('sex')['d_x'].cumsum()

# 4. Shift that down by 1 within each sex group so Age 0 starts with 0 deaths before it
df_both['deaths_before'] = df_both.groupby('sex')['cum_deaths'].shift(1).fillna(0)

# 5. Survivorship Math
df_both['l_x_raw'] = df_both['n_0'] - df_both['deaths_before']
df_both['l_x_100k'] = (df_both['l_x_raw'] / df_both['n_0']) * 100000

fig = px.line(
    df_both,
    x='x',
    y='l_x_100k',
    color='sex',
    title='2024 Comparative Survivorship Curve (City of Manila)',
    labels={
        'x': 'Age Interval',
        'l_x_100k': 'Survivors (per 100,000) - Log Scale',
        'sex': 'Sex'
    },
    markers=True,
    log_y=True,
    color_discrete_map={"Male": "royalblue", "Female": "red"}
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45
)

fig.show()

## City of Davao Cohort Survivorship (for comparison)

In [7]:
query_davao = """
              with bucketed as ( select
                                     -- Age 0 falls below 1, so it automatically goes to Bucket 0.
                                     -- Ages 1-5 go to Bucket 1, 6-10 to Bucket 2, etc.
                                     width_bucket(age_years::int, 1, 121, 24) as sort_index
                                 from vw_vsr_deaths_joined
                                 where death_year = 2024
                                   and age_years is not null
                                   and pod_level = 'City'
                                   and pod_name = 'City of Davao' )
              select sort_index
                   , case when sort_index = 0 then '0'
                          else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text end as x
                   , count(*)                                                                                  as d_x
              from bucketed
              group by 1, 2
              order by 1 asc; \
              """

with engine.connect() as conn:
    df_davao = pd.read_sql_query(text(query_davao), conn)

n_0 = df_davao['d_x'].sum()

# l_x (Surviving)
df_davao['l_x_raw'] = n_0 - df_davao['d_x'].cumsum().shift(1, fill_value=0)

# Survivorship (l_x / n0) - formatting as a percentage string to match the worksheet
df_davao['l_x_percent'] = ((df_davao['l_x_raw'] / n_0) * 100).round(2).astype(str) + '%'

# q_x (Age-specific mortality) - formatting as a percentage string
df_davao['q_x'] = ((df_davao['d_x'] / df_davao['l_x_raw']) * 100).round(2).astype(str) + '%'

# L_x = (l_x + l_x+1) / 2
df_davao['l_x_next'] = df_davao['l_x_raw'].shift(-1, fill_value=0)
df_davao['L_x'] = (df_davao['l_x_raw'] + df_davao['l_x_next']) / 2

# T_x = Sum of L_x from age x to last age
df_davao['T_x'] = df_davao['L_x'].iloc[::-1].cumsum().iloc[::-1]

# e_x = T_x / L_x (Life Expectancy)
df_davao['e_x'] = (df_davao['T_x'] / df_davao['L_x']).round(5)

# Format to match the worksheet columns perfectly and drop the sorting index
final_table_davao = df_davao[['x', 'd_x', 'l_x_percent', 'l_x_raw', 'q_x', 'L_x', 'T_x', 'e_x']]
print(final_table_davao.to_string(index=False))

     x  d_x l_x_percent  l_x_raw    q_x     L_x      T_x      e_x
     0  388      100.0%    12929   3.0% 12735.0 163707.5 12.85493
   1-5  165       97.0%    12541  1.32% 12458.5 150972.5 12.11803
  6-10   91      95.72%    12376  0.74% 12330.5 138514.0 11.23345
 11-15  103      95.02%    12285  0.84% 12233.5 126183.5 10.31459
 16-20  194      94.22%    12182  1.59% 12085.0 113950.0  9.42904
 21-25  232      92.72%    11988  1.94% 11872.0 101865.0  8.58027
 26-30  298      90.93%    11756  2.53% 11607.0  89993.0  7.75334
 31-35  353      88.62%    11458  3.08% 11281.5  78386.0  6.94819
 36-40  454      85.89%    11105  4.09% 10878.0  67104.5  6.16883
 41-45  579      82.38%    10651  5.44% 10361.5  56226.5  5.42648
 46-50  730       77.9%    10072  7.25%  9707.0  45865.0  4.72494
 51-55  959      72.26%     9342 10.27%  8862.5  36158.0  4.07989
 56-60 1202      64.84%     8383 14.34%  7782.0  27295.5  3.50752
 61-65 1355      55.54%     7181 18.87%  6503.5  19513.5  3.00046
 66-70 154

In [18]:
# We use the raw survivor count ('l_x_raw') for the Y-axis
fig = px.line(
    final_table_davao,
    x='x',  # The age brackets (0, 1-5, 6-10...)
    y='l_x_raw',  # The remaining survivors out of your starting population
    title='2024 Cohort Survivorship Curve (City of Davao)',
    labels={
        'x': 'Age Interval',
        'l_x_raw': 'Number of Survivors (lx)'
    },
    log_y=True,
    markers=True  # Adds distinct dots to each data point
)

# Standard formatting to make it look professional
fig.update_traces(fill='tozeroy', line_color='indigo')
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45  # Tilts the age labels so they don't overlap
)

fig.show()

In [23]:
query_rural = """
              with region_filtered as (
                  select
                      case
                          when pod_psgc like '13806%' then 'Manila (Urban)'
                          when pod_psgc like '08026%' then 'Eastern Samar (Rural)'
                          else null
                          end as region,
                      age_years::int as age_int
                  from vw_vsr_deaths_joined
                  where death_year = 2024
                    and age_years is not null
              ),
                  bucketed as (
                      select
                          region,
                          width_bucket(age_int, 1, 101, 20) as sort_index
                      from region_filtered
                      where region is not null
                  )
              select
                  region,
                  sort_index,
                  case
                      when sort_index = 0 then '0'
                      else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text
                      end as x,
                  count(*) as d_x
              from bucketed
              group by 1, 2, 3
              order by 1, 2 asc; \
              """

with engine.connect() as conn:
    df_rural = pd.read_sql_query(text(query_rural), conn)

# 2. Vectorized Pandas Math
df_rural = df_rural.sort_values(by=['region', 'sort_index'])

# Total starting population per region
df_rural['n_0'] = df_rural.groupby('region')['d_x'].transform('sum')

# Cumulative deaths and shifted deaths
df_rural['cum_deaths'] = df_rural.groupby('region')['d_x'].cumsum()
df_rural['deaths_before'] = df_rural.groupby('region')['cum_deaths'].shift(1).fillna(0)

# Survivorship and 100k Normalization
df_rural['l_x_raw'] = df_rural['n_0'] - df_rural['deaths_before']
df_rural['l_x_100k'] = (df_rural['l_x_raw'] / df_rural['n_0']) * 100000

# 3. Plotting the Comparison
fig = px.line(
    df_rural,
    x='x',
    y='l_x_100k',
    color='region',
    title='2024 Urban vs. Rural Survivorship Curve (Manila vs. Eastern Samar)',
    labels={
        'x': 'Age Interval',
        'l_x_100k': 'Survivors (per 100,000) - Log Scale',
        'region': 'Geography'
    },
    markers=True,
    log_y=True,
    color_discrete_map={"Manila (Urban)": "darkorange", "Eastern Samar (Rural)": "forestgreen"}
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45
)

fig.show()

## Nationwide Cohort Survivorship

In [9]:
query = """
        with bucketed as ( select
                               -- Age 0 falls below 1, so it automatically goes to Bucket 0.
                               -- Ages 1-5 go to Bucket 1, 6-10 to Bucket 2, etc.
                               width_bucket(age_years::int, 1, 121, 24) as sort_index
                           from vw_vsr_deaths_joined
                           where death_year = 2024
                             and age_years is not null )
        select sort_index
             , case when sort_index = 0 then '0'
                    else ((sort_index - 1) * 5 + 1)::text || '-' || ((sort_index - 1) * 5 + 5)::text end as x
             , count(*)                                                                                  as d_x
        from bucketed
        group by 1, 2
        order by 1 asc; \
        """

with engine.connect() as conn:
    df = pd.read_sql_query(text(query), conn)

n_0 = df['d_x'].sum()

# l_x (Surviving)
df['l_x_raw'] = n_0 - df['d_x'].cumsum().shift(1, fill_value=0)

# Survivorship (l_x / n0) - formatting as a percentage string to match the worksheet
df['l_x_percent'] = ((df['l_x_raw'] / n_0) * 100).round(2).astype(str) + '%'

# q_x (Age-specific mortality) - formatting as a percentage string
df['q_x'] = ((df['d_x'] / df['l_x_raw']) * 100).round(2).astype(str) + '%'

# L_x = (l_x + l_x+1) / 2
df['l_x_next'] = df['l_x_raw'].shift(-1, fill_value=0)
df['L_x'] = (df['l_x_raw'] + df['l_x_next']) / 2

# T_x = Sum of L_x from age x to last age
df['T_x'] = df['L_x'].iloc[::-1].cumsum().iloc[::-1]

# e_x = T_x / L_x (Life Expectancy)
df['e_x'] = (df['T_x'] / df['L_x']).round(5)

# Format to match the worksheet columns perfectly and drop the sorting index
final_table = df[['x', 'd_x', 'l_x_percent', 'l_x_raw', 'q_x', 'L_x', 'T_x', 'e_x']]
print(final_table.to_string(index=False))

     x   d_x l_x_percent  l_x_raw    q_x      L_x       T_x      e_x
     0 21104      100.0%   701811  3.01% 691259.0 9152996.5 13.24105
   1-5  8170      96.99%   680707   1.2% 676622.0 8461737.5 12.50586
  6-10  4379      95.83%   672537  0.65% 670347.5 7785115.5 11.61355
 11-15  5415       95.2%   668158  0.81% 665450.5 7114768.0 10.69166
 16-20  9059      94.43%   662743  1.37% 658213.5 6449317.5  9.79822
 21-25 11484      93.14%   653684  1.76% 647942.0 5791104.0  8.93769
 26-30 14205      91.51%   642200  2.21% 635097.5 5143162.0  8.09822
 31-35 17472      89.48%   627995  2.78% 619259.0 4508064.5  7.27977
 36-40 21479      86.99%   610523  3.52% 599783.5 3888805.5  6.48368
 41-45 29472      83.93%   589044   5.0% 574308.0 3289022.0  5.72693
 46-50 37161      79.73%   559572  6.64% 540991.5 2714714.0  5.01803
 51-55 48958      74.44%   522411  9.37% 497932.0 2173722.5  4.36550
 56-60 58113      67.46%   473453 12.27% 444396.5 1675790.5  3.77094
 61-65 69122      59.18%   415340 

In [10]:
# We use the raw survivor count ('l_x_raw') for the Y-axis
fig = px.line(
    final_table,
    x='x',  # The age brackets (0, 1-5, 6-10...)
    y='l_x_raw',  # The remaining survivors out of your starting population
    title='2024 Nationwide Cohort Survivorship Curve (Male & Female)',
    labels={
        'x': 'Age Interval',
        'l_x_raw': 'Number of Survivors (lx)'
    },
    log_y=True,
    markers=True  # Adds distinct dots to each data point
)

# Standard formatting to make it look professional
fig.update_traces(fill='tozeroy', line_color='indigo')
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    xaxis_tickangle=-45  # Tilts the age labels so they don't overlap
)

fig.show()

## Mortality Statistics

### Historical Area Graph


In [11]:
query = """
        select death_year, death_month, sex, count(*) as total_deaths
        from vw_vsr_deaths_joined vvdj
        where death_year between 2006 and 2024
          and death_month between 1 and 12
        group by 1, 2, 3
        order by 1, 2 asc \
        """

with engine.connect() as conn:
    df = pd.read_sql_query(query, conn)

df['date'] = pd.to_datetime(df['death_year'].astype(str) + '-' + df['death_month'].astype(str) + '-01')

fig = px.area(df, x="date", y="total_deaths", color="sex",
              title="Monthly Mortality in the Philippines by Sex (2006-2024)",
              labels={"date": "Timeline", "total_deaths": "Total Recorded Deaths", "sex": "Sex"})

fig.show()

### Leading Causes of Death

In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import text

# 1. SQL for Manila's Top Causes (2024)
query_manila = """
               with categorized as ( select case when icd10_code like 'I2%' then 'Ischemic Heart Disease'
                                                 when icd10_code like 'I6%' then 'Stroke (Cerebrovascular)'
                                                 when icd10_code like 'C%' then 'Cancer (Neoplasms)'
                                                 when icd10_code like 'J1%' then 'Pneumonia'
                                                 when icd10_code like 'E1%' then 'Diabetes Mellitus'
                                                 when icd10_code = 'A15' or icd10_code = 'A16' or icd10_code = 'A17' or
                                                      icd10_code = 'A18' or icd10_code = 'A19' then 'Tuberculosis'
                                                 when icd10_code like 'I1%' then 'Hypertension'
                                                 else 'Other Causes' end as cause_group
                                     from vw_vsr_deaths_joined
                                     where death_year = 2024
                                       and pod_psgc like '13806%' )
               select cause_group, count(*) as total_deaths
               from categorized
               group by 1
               order by 2 desc; \
               """

# 2. SQL for Nationwide Top Causes (2024)
query_nationwide = """
                   with categorized as ( select case when icd10_code like 'I2%' then 'Ischemic Heart Disease'
                                                     when icd10_code like 'I6%' then 'Stroke (Cerebrovascular)'
                                                     when icd10_code like 'C%' then 'Cancer (Neoplasms)'
                                                     when icd10_code like 'J1%' then 'Pneumonia'
                                                     when icd10_code like 'E1%' then 'Diabetes Mellitus'
                                                     when icd10_code = 'A15' or icd10_code = 'A16' or
                                                          icd10_code = 'A17' or icd10_code = 'A18' or icd10_code = 'A19'
                                                         then 'Tuberculosis'
                                                     when icd10_code like 'I1%' then 'Hypertension'
                                                     else 'Other Causes' end as cause_group
                                         from vw_vsr_deaths_joined
                                         where death_year = 2024 )
                   select cause_group, count(*) as total_deaths
                   from categorized
                   group by 1
                   order by 2 desc; \
                   """

# Fetch the data
with engine.connect() as conn:
    df_manila = pd.read_sql_query(text(query_manila), conn)
    df_nation = pd.read_sql_query(text(query_nationwide), conn)

# 3. Create the Subplot Canvas
# 'domain' type is required specifically for Pie/Donut charts in Plotly
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['City of Manila', 'Nationwide']
)

# 4. Add the Manila Trace (Left)
fig.add_trace(
    go.Pie(
        labels=df_manila['cause_group'],
        values=df_manila['total_deaths'],
        name="Manila",
        textinfo='percent+label',
    ),
    row=1, col=1
)

# 5. Add the Nationwide Trace (Right)
fig.add_trace(
    go.Pie(
        labels=df_nation['cause_group'],
        values=df_nation['total_deaths'],
        name="Nationwide",
        textinfo='percent+label',
    ),
    row=1, col=2
)

# 6. Transform into Donut Charts and Add Center Annotations
fig.update_traces(hole=0.45, hoverinfo="label+percent+name+value")

fig.update_layout(
    title_text="Leading Causes of Mortality: Manila vs. Nationwide (2024)",
    title_x=0.5,  # Centers the main title
    annotations=[
        # Left Center Text
        dict(text='Manila', x=0.225, y=0.5, font_size=16, showarrow=False, xanchor="center"),
        # Right Center Text
        dict(text='PH', x=0.775, y=0.5, font_size=16, showarrow=False, xanchor="center")
    ]
)

fig.show()

In [13]:
query_manila = """
               with categorized as ( select case when icd10_code like 'I2%' then 'Ischemic Heart Disease'
                                                 when icd10_code like 'I6%' then 'Stroke (Cerebrovascular)'
                                                 when icd10_code like 'C%' then 'Cancer (Neoplasms)'
                                                 when icd10_code like 'J1%' then 'Pneumonia'
                                                 when icd10_code like 'E1%' then 'Diabetes Mellitus'
                                                 when icd10_code = 'A15' or icd10_code = 'A16' or icd10_code = 'A17' or
                                                      icd10_code = 'A18' or icd10_code = 'A19' then 'Tuberculosis'
                                                 when icd10_code like 'I1%' then 'Hypertension'
                                                 else 'Other Causes' end as cause_group
                                     from vw_vsr_deaths_joined
                                     where pod_psgc like '13806%' )
               select cause_group, count(*) as total_deaths
               from categorized
               group by 1
               order by 2 desc; \
               """

query_nationwide = """
                   with categorized as ( select case when icd10_code like 'I2%' then 'Ischemic Heart Disease'
                                                     when icd10_code like 'I6%' then 'Stroke (Cerebrovascular)'
                                                     when icd10_code like 'C%' then 'Cancer (Neoplasms)'
                                                     when icd10_code like 'J1%' then 'Pneumonia'
                                                     when icd10_code like 'E1%' then 'Diabetes Mellitus'
                                                     when icd10_code = 'A15' or icd10_code = 'A16' or
                                                          icd10_code = 'A17' or icd10_code = 'A18' or icd10_code = 'A19'
                                                         then 'Tuberculosis'
                                                     when icd10_code like 'I1%' then 'Hypertension'
                                                     else 'Other Causes' end as cause_group
                                         from vw_vsr_deaths_joined )
                   select cause_group, count(*) as total_deaths
                   from categorized
                   group by 1
                   order by 2 desc; \
                   """

# Fetch the data
with engine.connect() as conn:
    df_manila = pd.read_sql_query(text(query_manila), conn)
    df_nation = pd.read_sql_query(text(query_nationwide), conn)

# 3. Create the Subplot Canvas
# 'domain' type is required specifically for Pie/Donut charts in Plotly
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['City of Manila', 'Nationwide']
)

# 4. Add the Manila Trace (Left)
fig.add_trace(
    go.Pie(
        labels=df_manila['cause_group'],
        values=df_manila['total_deaths'],
        name="Manila",
        textinfo='percent+label',
    ),
    row=1, col=1
)

# 5. Add the Nationwide Trace (Right)
fig.add_trace(
    go.Pie(
        labels=df_nation['cause_group'],
        values=df_nation['total_deaths'],
        name="Nationwide",
        textinfo='percent+label',
    ),
    row=1, col=2
)

# 6. Transform into Donut Charts and Add Center Annotations
fig.update_traces(hole=0.45, hoverinfo="label+percent+name+value")

fig.update_layout(
    title_text="Leading Causes of Mortality: Manila vs. Nationwide (2006-2024)",
    title_x=0.5,  # Centers the main title
    annotations=[
        # Left Center Text
        dict(text='Manila', x=0.225, y=0.5, font_size=16, showarrow=False, xanchor="center"),
        # Right Center Text
        dict(text='PH', x=0.775, y=0.5, font_size=16, showarrow=False, xanchor="center")
    ]
)

fig.show()

### Tokhang

In [14]:
from datetime import datetime

# 1. SQL for the Tokhang Hypothesis
query_tokhang = """
                with tokhang_data as ( select death_year
                                            , death_month
                                            , case when (icd10_code >= 'X85' and icd10_code < 'Y10') or icd10_code like 'Y35%'
                                                       then 'Assault & Legal Intervention'
                                                   else null end as event_category
                                       from vw_vsr_deaths_joined
                                       where death_year >= 2010
                                         and death_month between 1 and 12 )
                select death_year, death_month, event_category, count(*) as total_deaths
                from tokhang_data
                where event_category is not null
                group by 1, 2, 3
                order by 1, 2 asc; \
                """

with engine.connect() as conn:
    df_tokhang = pd.read_sql_query(text(query_tokhang), conn)

df_tokhang['date'] = pd.to_datetime(
    df_tokhang['death_year'].astype(str) + '-' + df_tokhang['death_month'].astype(str) + '-01')

# 2. Render the Tokhang Line Chart
fig1 = px.area(
    df_tokhang,
    x='date',
    y='total_deaths',
    color='event_category',
    title='Oplan Tokhang\'s Effect on Deaths',
    labels={'date': 'Timeline', 'total_deaths': 'Total Monthly Deaths', 'event_category': 'Cause of Death'},
    color_discrete_map={
        'Assault & Legal Intervention': 'darkred',
    }
)

tokhang_start = datetime(2016, 7, 1).timestamp() * 1000

fig1.add_vline(
    x=tokhang_start,
    line_width=2, line_dash="dash", line_color="black",
    annotation_text="Oplan Tokhang Initiated", annotation_position="top right"
)

fig1.update_layout(template="plotly_white", hovermode="x unified")
fig1.show()

### Ecological Vectors

In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

query_ecological = """
                   with eco_data as ( select death_year
                                           , death_month
                                           , case
                           -- Zoonotic & Vector-Borne
                           when icd10_code like 'A90%' or icd10_code like 'A91%' or icd10_code like 'A97%'
                               then 'Dengue Fever'
                           when icd10_code like 'A82%' then 'Rabies'
                           when icd10_code like 'A27%' then 'Leptospirosis'

                           -- Endemic Respiratory
                           when icd10_code like 'J1%' then 'Pneumonia'
                           when icd10_code like 'A15%' or icd10_code like 'A16%' then 'Tuberculosis'

                           else null end as event_category
                                      from vw_vsr_deaths_joined
                                      where death_year >= 2006
                                        and death_month between 1 and 12 )
                   select death_year, death_month, event_category, count(*) as total_deaths
                   from eco_data
                   where event_category is not null
                   group by 1, 2, 3
                   order by 1, 2 asc; \
                   """

with engine.connect() as conn:
    df_eco = pd.read_sql_query(text(query_ecological), conn)

df_eco['date'] = pd.to_datetime(df_eco['death_year'].astype(str) + '-' + df_eco['death_month'].astype(str) + '-01')

# 1. Create a 2-row subplot canvas that shares the same X-axis (Timeline)
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Endemic Respiratory Load",
        "Zoonotic & Vector-Borne Load"
    )
)

# Define the disease groups
respiratory = ['Pneumonia', 'Tuberculosis']
vector_borne = ['Dengue Fever', 'Leptospirosis', 'Rabies']

# Set specific colors so they match your previous visual language
colors = {
    'Tuberculosis': '#f5b041', 'Pneumonia': '#a569bd',
    'Dengue Fever': '#5dade2', 'Leptospirosis': '#ec7063', 'Rabies': '#48c9b0'
}

# 2. Loop through and plot the Respiratory diseases on the Top Row (Row 1)
for disease in respiratory:
    df_filtered = df_eco[df_eco['event_category'] == disease]
    fig.add_trace(
        go.Scatter(
            x=df_filtered['date'], y=df_filtered['total_deaths'],
            name=disease, mode='lines', fill='tozeroy',
            line=dict(color=colors[disease])
        ),
        row=1, col=1
    )

# 3. Loop through and plot Vector/Zoonotic diseases on the Bottom Row (Row 2)
for disease in vector_borne:
    df_filtered = df_eco[df_eco['event_category'] == disease]
    fig.add_trace(
        go.Scatter(
            x=df_filtered['date'], y=df_filtered['total_deaths'],
            name=disease, mode='lines', fill='tozeroy',
            line=dict(color=colors[disease])
        ),
        row=2, col=1
    )

# 4. Clean up the layout
fig.update_layout(
    title_text="Ecological & Endemic Mortality Vectors in the Philippines (2006-2024)",
    height=700,  # Make the chart a bit taller to fit both rows comfortably
    template="plotly_white",
    hovermode="x unified"
)

# Update the Y-axis labels for clarity
fig.update_yaxes(title_text="Monthly Deaths", row=1, col=1)
fig.update_yaxes(title_text="Monthly Deaths", row=2, col=1)

fig.show()